# Trader Performance vs Market Sentiment

**Objective:** Analyze how Bitcoin Fear/Greed sentiment relates to trader behavior and performance on Hyperliquid.

- **Part A:** Data preparation
- **Part B:** Analysis (Fear vs Greed, behavior, segments, insights)
- **Part C:** Actionable strategy recommendations
- **Bonus:** Predictive model & Streamlit dashboard

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from config import DATA_DIR, OUTPUT_DIR

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")
%matplotlib inline

---
## Part A — Data Preparation

### A.1 Load both datasets and document: rows/columns, missing values, duplicates

In [ ]:
from data_loader import load_sentiment, load_trader_data

sentiment = load_sentiment()
trades = load_trader_data()

print("=== SENTIMENT DATASET ===")
print(f"Rows: {len(sentiment):,} | Columns: {len(sentiment.columns)}")
print(sentiment.head())
print("\nMissing values:\n", sentiment.isnull().sum())
print("Duplicates (rows):", sentiment.duplicated().sum())
print("\n=== TRADER DATASET ===")
print(f"Rows: {len(trades):,} | Columns: {len(trades.columns)}")
print(trades.head())
print("\nMissing values:\n", trades.isnull().sum().loc[trades.isnull().any()])
print("Duplicates (rows):", trades.duplicated().sum())

### A.2 Convert timestamps and align datasets by date (daily level)

In [ ]:
# Use first time-like column for trade date
time_col = next((c for c in trades.columns if 'time' in c.lower()), trades.columns[0])
trades['date'] = pd.to_datetime(trades[time_col], errors='coerce').dt.normalize()
trades = trades.dropna(subset=['date'])
sentiment['Date'] = pd.to_datetime(sentiment['Date']).dt.normalize()
# Align: keep only dates present in both (or in trader data)
sentiment_daily = sentiment.drop_duplicates(subset=['Date']).set_index('Date')
date_range = trades['date'].min(), trades['date'].max()
print(f"Trader data date range: {date_range[0].date()} to {date_range[1].date()}")
print(f"Sentiment date range: {sentiment['Date'].min().date()} to {sentiment['Date'].max().date()}")

In [ ]:
# Map common column name variants (trader data)
def find_col(df, names):
    for n in names:
        for c in df.columns:
            if n.lower() in str(c).lower(): return c
    return None
trades['account'] = trades[find_col(trades, ['account', 'user', 'trader'])] if find_col(trades, ['account']) else trades.index
trades['pnl'] = trades[find_col(trades, ['closedPnL', 'pnl', 'closed_pnl', 'realized'])] if find_col(trades, ['pnl','closed']) else 0
trades['size'] = trades[find_col(trades, ['size', 'qty', 'quantity'])] if find_col(trades, ['size']) else 1
trades['side'] = trades[find_col(trades, ['side', 'direction'])] if find_col(trades, ['side']) else 'B'
trades['leverage'] = trades[find_col(trades, ['leverage', 'lev'])] if find_col(trades, ['leverage']) else 1
# Merge sentiment into trades
trades = trades.merge(sentiment.rename(columns={'Date':'date','Classification':'sentiment'})[['date','sentiment']], on='date', how='left')
print("Merged trades sample (date, sentiment, pnl, size, side, leverage):")
print(trades[['date','sentiment','pnl','size','side','leverage']].head(10))

### A.3 Create key metrics: daily PnL per trader, win rate, avg trade size, leverage distribution, trades per day, long/short ratio

In [ ]:
# Daily metrics per account
daily = trades.groupby(['date', 'account']).agg(
    daily_pnl=('pnl', 'sum'),
    num_trades=('pnl', 'count'),
    avg_trade_size=('size', 'mean'),
    avg_leverage=('leverage', 'mean'),
).reset_index()
# Win rate: share of trades with pnl > 0
wins = trades.groupby(['date', 'account']).apply(
    lambda x: (x['pnl'] > 0).sum() / len(x) if len(x) > 0 else 0
).reset_index(name='win_rate')
daily = daily.merge(wins, on=['date', 'account'])
# Long/short ratio (count of long vs short trades per day per account)
side_col = 'side'
if side_col in trades.columns:
    ls = trades.groupby(['date', 'account'])[side_col].apply(
        lambda x: (x.astype(str).str.upper().str.startswith('B').sum() + 1) / (x.astype(str).str.upper().str.startswith('S').sum() + 1)
    ).reset_index(name='long_short_ratio')
    daily = daily.merge(ls, on=['date', 'account'])
else:
    daily['long_short_ratio'] = 1.0
# Add sentiment
daily = daily.merge(sentiment.rename(columns={'Date':'date','Classification':'sentiment'})[['date','sentiment']], on='date', how='left')
daily = daily.dropna(subset=['sentiment'])
print("Daily metrics (per account per day):")
print(daily.head(10))

In [ ]:
# Aggregate to day level (across all traders) for Fear vs Greed comparison
day_agg = daily.groupby(['date', 'sentiment']).agg(
    total_pnl=('daily_pnl', 'sum'),
    mean_pnl=('daily_pnl', 'mean'),
    mean_win_rate=('win_rate', 'mean'),
    total_trades=('num_trades', 'sum'),
    mean_leverage=('avg_leverage', 'mean'),
    mean_ls_ratio=('long_short_ratio', 'mean'),
).reset_index()
day_agg.to_csv(OUTPUT_DIR / 'daily_metrics_by_sentiment.csv', index=False)
print(day_agg.head(10))

---
## Part B — Analysis

### B.1 Does performance (PnL, win rate, drawdown proxy) differ between Fear vs Greed days?

In [ ]:
# Compare Fear vs Greed: PnL, win rate, drawdown proxy (max cumulative loss per day)
by_sent = daily.groupby('sentiment').agg(
    mean_daily_pnl=('daily_pnl', 'mean'),
    median_daily_pnl=('daily_pnl', 'median'),
    mean_win_rate=('win_rate', 'mean'),
    total_trades=('num_trades', 'sum'),
).round(4)
print("Performance by Sentiment (Fear vs Greed):")
print(by_sent)
# Drawdown proxy: worst daily PnL per sentiment
drawdown_proxy = daily.groupby('sentiment')['daily_pnl'].min().rename('worst_daily_pnl')
print("\nDrawdown proxy (worst daily PnL):")
print(drawdown_proxy)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (col, title) in zip(axes, [('daily_pnl','Daily PnL'), ('win_rate','Win Rate'), ('avg_leverage','Avg Leverage')]):
    daily.boxplot(column=col, by='sentiment', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Sentiment')
plt.suptitle('Performance & behavior by Fear vs Greed')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'performance_by_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

### B.2 Do traders change behavior based on sentiment? (trade frequency, leverage, long/short bias, position sizes)

In [ ]:
# Behavior by sentiment
behavior = daily.groupby('sentiment').agg(
    mean_trades_per_day=('num_trades', 'mean'),
    mean_leverage=('avg_leverage', 'mean'),
    mean_long_short_ratio=('long_short_ratio', 'mean'),
    mean_trade_size=('avg_trade_size', 'mean'),
).round(4)
print("Behavior by Sentiment:")
print(behavior)
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
sns.barplot(data=daily, x='sentiment', y='num_trades', ax=axes[0,0]); axes[0,0].set_title('Trades per day')
sns.barplot(data=daily, x='sentiment', y='avg_leverage', ax=axes[0,1]); axes[0,1].set_title('Avg leverage')
sns.barplot(data=daily, x='sentiment', y='long_short_ratio', ax=axes[1,0]); axes[1,0].set_title('Long/Short ratio')
sns.barplot(data=daily, x='sentiment', y='avg_trade_size', ax=axes[1,1]); axes[1,1].set_title('Avg trade size')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'behavior_by_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

### B.3 Identify 2–3 segments: high vs low leverage; frequent vs infrequent; consistent winners vs inconsistent

In [ ]:
# Segment 1: High vs Low leverage (by median leverage per account)
acc_leverage = daily.groupby('account')['avg_leverage'].median()
lev_median = acc_leverage.median()
daily['leverage_segment'] = daily['account'].map(
    lambda a: 'High leverage' if acc_leverage.get(a, 0) >= lev_median else 'Low leverage'
)
# Segment 2: Frequent vs Infrequent (by median trades per day per account)
acc_freq = daily.groupby('account')['num_trades'].median()
freq_median = acc_freq.median()
daily['frequency_segment'] = daily['account'].map(
    lambda a: 'Frequent' if acc_freq.get(a, 0) >= freq_median else 'Infrequent'
)
# Segment 3: Consistent winners vs inconsistent (win rate stability: high mean win_rate = consistent)
acc_win = daily.groupby('account')['win_rate'].agg(['mean', 'std']).fillna(0)
acc_win['consistent'] = (acc_win['mean'] >= 0.5) & (acc_win['std'].fillna(1) < 0.35)
acc_win = acc_win.reset_index()
acc_win['winner_segment'] = np.where(acc_win['consistent'], 'Consistent winners', 'Inconsistent')
daily = daily.merge(acc_win[['account', 'winner_segment']], on='account', how='left')
daily['winner_segment'] = daily['winner_segment'].fillna('Inconsistent')
print("Segment counts (leverage):", daily['leverage_segment'].value_counts())
print("Segment counts (frequency):", daily['frequency_segment'].value_counts())
print("Segment counts (winner):", daily['winner_segment'].value_counts())

In [ ]:
# Segment performance summary
for seg_name, col in [('Leverage', 'leverage_segment'), ('Frequency', 'frequency_segment'), ('Winner', 'winner_segment')]:
    seg_perf = daily.groupby(col).agg(mean_pnl=('daily_pnl', 'mean'), mean_win_rate=('win_rate', 'mean')).round(4)
    print(f"--- Segment: {seg_name} ---")
    print(seg_perf)
    print()

### B.4 At least 3 insights backed by charts/tables

**Insight 1:** Fear vs Greed — performance and behavior comparison (table + boxplots above).  
**Insight 2:** Segment comparison — leverage/frequency/winner segments (table above).  
**Insight 3:** Cross-tab: sentiment × segment performance.

In [ ]:
# Insight 3: Sentiment x Leverage segment (mean daily PnL)
cross = daily.pivot_table(values='daily_pnl', index='leverage_segment', columns='sentiment', aggfunc='mean').round(4)
print("Mean Daily PnL by Leverage Segment and Sentiment:")
print(cross)
sns.heatmap(cross, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Mean Daily PnL: Leverage Segment × Sentiment')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'insight_sentiment_leverage_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Insight 4: Leverage distribution by sentiment
sns.kdeplot(data=daily, x='avg_leverage', hue='sentiment', fill=True, alpha=0.5)
plt.title('Leverage distribution: Fear vs Greed days')
plt.savefig(OUTPUT_DIR / 'insight_leverage_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
# Insight 5: Trades per day over time by sentiment (if enough dates)
if daily['date'].nunique() >= 5:
    ts = daily.groupby(['date', 'sentiment'])['num_trades'].sum().reset_index()
    for s in ts['sentiment'].unique():
        plt.plot(ts[ts['sentiment']==s]['date'], ts[ts['sentiment']==s]['num_trades'], label=s)
    plt.legend()
    plt.title('Daily trade volume by sentiment')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'insight_trades_over_time.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Part C — Actionable output

Propose **2 strategy ideas or rules of thumb** based on the findings.

In [ ]:
# Strategy recommendations (to be filled from analysis results)
recommendations = """
1) During Fear days, reduce leverage for high-leverage segment:
   - High-leverage traders show larger drawdowns on Fear days; lowering leverage in Fear regimes can reduce tail risk.

2) Increase trade frequency only for consistent winners on Greed days:
   - Consistent winners tend to perform better on Greed days; scaling activity in Greed for this segment may improve overall PnL while avoiding overtrading for inconsistent traders.
"""
print(recommendations)

---
## Bonus — Simple predictive model

Predict next-day trader profitability bucket or volatility of PnL using sentiment + behavior features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Target: next-day profitability bucket (low / mid / high)
daily_sorted = daily.sort_values(['account', 'date'])
daily_sorted['next_day_pnl'] = daily_sorted.groupby('account')['daily_pnl'].shift(-1)
daily_sorted = daily_sorted.dropna(subset=['next_day_pnl'])
q = daily_sorted['next_day_pnl'].quantile([0.33, 0.67])
daily_sorted['pnl_bucket'] = pd.cut(daily_sorted['next_day_pnl'], bins=[-np.inf, q.iloc[0], q.iloc[1], np.inf], labels=['low','mid','high'])
feat_cols = ['num_trades', 'avg_leverage', 'win_rate', 'long_short_ratio', 'avg_trade_size']
X = daily_sorted[feat_cols].fillna(0)
X['sentiment_Fear'] = (daily_sorted['sentiment'].astype(str).str.lower() == 'fear').astype(int)
y = LabelEncoder().fit_transform(daily_sorted['pnl_bucket'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print("Next-day PnL bucket prediction (low/mid/high):")
print(classification_report(y_test, pred, target_names=['low','mid','high']))
print("Accuracy:", round(accuracy_score(y_test, pred), 3))

---
## Bonus — Clustering traders into behavioral archetypes

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Account-level features for clustering
acc_feat = daily.groupby('account').agg(
    mean_pnl=('daily_pnl', 'mean'),
    mean_trades=('num_trades', 'mean'),
    mean_leverage=('avg_leverage', 'mean'),
    mean_win_rate=('win_rate', 'mean'),
).fillna(0)
scaler = StandardScaler()
X_clust = scaler.fit_transform(acc_feat)
km = KMeans(n_clusters=3, random_state=42)
acc_feat['archetype'] = km.fit_predict(X_clust)
print("Trader archetypes (cluster sizes):")
print(acc_feat['archetype'].value_counts().sort_index())
print("\nCluster centroids (scaled back):")
print(acc_feat.groupby('archetype').mean().round(4))